# 02 EDA: 월별 부동산 거래 동향 분석

`data/processed/monthly_merged.csv`를 기준으로 지역별 가격 수준, 월별 가격 변화, 거래량 변화, 월세 거래 비중을 확인한다. 3주차 피드백에 맞춰 그래프 제목, 축 이름, 단위를 통일하고 보고서/PPT에 사용할 수 있는 간단한 해석을 함께 정리했다.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# 노트북을 repo root, notebooks 폴더, 또는 상위 작업 폴더 어디서 실행해도 동작하도록 경로를 맞춘다.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_CANDIDATES = [
    ROOT / "data" / "processed" / "monthly_merged.csv",
    ROOT.parent / "data" / "processed" / "monthly_merged.csv",
    Path.cwd() / "data" / "processed" / "monthly_merged.csv",
    Path.cwd().parent / "data" / "processed" / "monthly_merged.csv",
]

DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    checked_paths = "\n".join(str(path) for path in DATA_CANDIDATES)
    raise FileNotFoundError(
        "monthly_merged.csv를 찾을 수 없습니다. 아래 경로 중 하나에 파일이 있어야 합니다:\n"
        + checked_paths
    )

OUTPUT_DIR = ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 150


def comma_axis(ax):
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{int(x):,}"))


def percent_axis(ax):
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x:.0f}%"))


def save_fig(filename):
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    plt.show()

In [ ]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df["contract_month"] = pd.to_datetime(df["contract_month"])

df.head()

## 1. 지역별 평균 매매가

In [ ]:
gu_sale = (
    df.groupby("gu")["avg_sale_price"]
    .mean()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 6))
gu_sale.plot(kind="bar", ax=ax, color="#4C78A8")
ax.set_title("지역별 평균 매매가", fontsize=16, pad=14)
ax.set_xlabel("자치구")
ax.set_ylabel("평균 매매가 (만원)")
ax.tick_params(axis="x", rotation=45)
comma_axis(ax)
ax.grid(axis="y", alpha=0.25)
save_fig("avg_sale_price_by_gu.png")

**해석**
- 강남구, 서초구, 용산구의 평균 매매가가 가장 높게 나타나 서울 내 고가 주거지역의 가격 수준이 뚜렷하게 확인된다.
- 도봉구, 강북구, 노원구, 금천구는 상대적으로 낮은 평균 매매가를 보여 지역 간 가격 격차가 크다.

## 2. 지역별 평균 전세 보증금

In [ ]:
gu_jeonse = (
    df.groupby("gu")["avg_jeonse_deposit"]
    .mean()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 6))
gu_jeonse.plot(kind="bar", ax=ax, color="#F58518")
ax.set_title("지역별 평균 전세 보증금", fontsize=16, pad=14)
ax.set_xlabel("자치구")
ax.set_ylabel("평균 전세 보증금 (만원)")
ax.tick_params(axis="x", rotation=45)
comma_axis(ax)
ax.grid(axis="y", alpha=0.25)
save_fig("avg_jeonse_deposit_by_gu.png")

**해석**
- 서초구, 강남구, 용산구의 평균 전세 보증금이 높게 나타나 매매가가 높은 지역에서 전세 가격도 높은 흐름을 보인다.
- 노원구, 도봉구, 금천구는 평균 전세 보증금이 낮아 전세 시장에서도 지역별 가격 차이가 확인된다.

## 3. 월별 평균 매매가 변화

In [ ]:
monthly_sale = (
    df.groupby("contract_month")["avg_sale_price"]
    .mean()
)

fig, ax = plt.subplots(figsize=(12, 5.5))
monthly_sale.plot(ax=ax, marker="o", linewidth=2, color="#4C78A8")
ax.set_title("월별 평균 매매가 변화", fontsize=16, pad=14)
ax.set_xlabel("계약월")
ax.set_ylabel("평균 매매가 (만원)")
comma_axis(ax)
ax.grid(alpha=0.25)
save_fig("monthly_sale_price_trend.png")

**해석**
- 서울 평균 매매가는 2023년 말 저점 이후 2025년 중반까지 상승하는 흐름을 보인다.
- 2026년 4월 평균 매매가는 2023년 5월보다 높은 수준으로, 분석 기간 전체로 보면 완만한 상승 추세가 확인된다.

## 4. 월별 평균 전세 보증금 변화

In [ ]:
monthly_jeonse = (
    df.groupby("contract_month")["avg_jeonse_deposit"]
    .mean()
)

fig, ax = plt.subplots(figsize=(12, 5.5))
monthly_jeonse.plot(ax=ax, marker="o", linewidth=2, color="#F58518")
ax.set_title("월별 평균 전세 보증금 변화", fontsize=16, pad=14)
ax.set_xlabel("계약월")
ax.set_ylabel("평균 전세 보증금 (만원)")
comma_axis(ax)
ax.grid(alpha=0.25)
save_fig("monthly_jeonse_deposit_trend.png")

**해석**
- 평균 전세 보증금은 2023년 5월 이후 전반적으로 상승하는 흐름을 보인다.
- 2025년 하반기에 높은 수준을 기록하며, 전세 부담이 점진적으로 커지는 모습이 확인된다.

## 5. 거래량 상위 5개 자치구의 월별 매매 거래량 변화

In [ ]:
top5_gu = (
    df.groupby("gu")["sale_count"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
)

fig, ax = plt.subplots(figsize=(12, 6))
for gu in top5_gu:
    gu_df = df[df["gu"] == gu].sort_values("contract_month")
    ax.plot(
        gu_df["contract_month"],
        gu_df["sale_count"],
        marker="o",
        linewidth=2,
        label=gu,
    )

ax.set_title("거래량 상위 5개 자치구의 월별 매매 거래량 변화", fontsize=16, pad=14)
ax.set_xlabel("계약월")
ax.set_ylabel("매매 거래량 (건)")
comma_axis(ax)
ax.grid(alpha=0.25)
ax.legend(title="자치구")
save_fig("monthly_sale_volume_top5_gu.png")

**해석**
- 분석 기간 누적 매매 거래량은 노원구, 송파구, 강동구, 강남구, 성북구가 높게 나타난다.
- 거래량은 특정 시점에 일시적으로 크게 증가하는 구간이 있어 가격 수준뿐 아니라 거래 활발도도 지역별로 차이가 있음을 보여준다.

## 6. 월별 월세 거래 비중 변화

In [ ]:
monthly_rent_ratio = (
    df.groupby("contract_month")[["monthly_count", "total_rent_count"]]
    .sum()
)
monthly_rent_ratio["monthly_ratio_pct"] = (
    monthly_rent_ratio["monthly_count"] / monthly_rent_ratio["total_rent_count"] * 100
)

fig, ax = plt.subplots(figsize=(12, 5.5))
monthly_rent_ratio["monthly_ratio_pct"].plot(
    ax=ax,
    marker="o",
    linewidth=2,
    color="#54A24B",
)
ax.set_title("월별 월세 거래 비중 변화", fontsize=16, pad=14)
ax.set_xlabel("계약월")
ax.set_ylabel("월세 거래 비중 (%)")
percent_axis(ax)
ax.grid(alpha=0.25)
save_fig("monthly_rent_ratio_trend.png")

**해석**
- 월세 거래 비중은 2023년 후반에 낮아졌다가 2026년 초에는 50% 안팎까지 올라가는 흐름을 보인다.
- 전월세 거래 중 월세가 차지하는 비중이 커지는 구간이 있어, 전세난·월세화 현상을 설명하는 보조 지표로 활용할 수 있다.